In [ ]:
!pip install -q aiohttp azure-storage-blob tqdm
import os
import json
import asyncio
import random
from typing import Set, List

import aiohttp
from aiohttp import TCPConnector, ClientTimeout
from tqdm import tqdm
from azure.storage.blob import BlobServiceClient, ContentSettings

# -----------------------
# CONFIG (COLAB SAFE)
# -----------------------
STREET_VIEW_KEY = os.environ.get("STREET_VIEW_API_KEY", "")
AZURE_CONN_STR = os.environ.get("AZURE_STORAGE_CONN_STR", "")

blob_path = "london"
prefix = "training"
container_name = blob_path

remote_meta_folder = "RandomSamplingMetadata56k"
target_folder = "training/Images56k"
checkpoint_blob_path = f"{prefix}/checkpoints/image_download_checkpoint.json"

angles = [0, 90, 180, 270]
image_size = "640x640"

# Colab-safe concurrency: controls simultaneous in-flight HTTP requests
MAX_CONCURRENT_DOWNLOADS = int(os.environ.get("MAX_CONCURRENT_DOWNLOADS", "75"))
REQUEST_TIMEOUT = int(os.environ.get("REQUEST_TIMEOUT", "20"))
MAX_RETRIES = int(os.environ.get("MAX_RETRIES", "5"))

# How many metadata files to process concurrently (each metadata starts up to 4 downloads)
NUM_META_WORKERS = int(os.environ.get("NUM_META_WORKERS", "10"))

checkpoint_file = "image_download_checkpoint.json"
SAVE_EVERY_N = 50  # save checkpoint every N processed metadata files

# -----------------------
# Azure setup (sync)
# -----------------------
blob_service = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
container_client = blob_service.get_container_client(container_name)

def upload_blob_sync(data: bytes, blob_path: str):
    """Synchronous upload — safe to call in run_in_executor."""
    try:
        blob = container_client.get_blob_client(blob_path)
        if blob.exists():
            return False
        blob.upload_blob(
            data,
            overwrite=False,
            content_settings=ContentSettings(content_type="image/jpeg"),
        )
        return True
    except Exception as e:
        # propagate exception to allow retry in caller if desired
        raise

# -----------------------
# Checkpoint helpers
# -----------------------
def load_checkpoint() -> Set[str]:
    blob = container_client.get_blob_client(checkpoint_blob_path)
    if not blob.exists():
        return set()

    data = blob.download_blob().readall()
    payload = json.loads(data)
    return set(payload.get("done", []))


def save_checkpoint(done: Set[str]):
    blob = container_client.get_blob_client(checkpoint_blob_path)
    payload = json.dumps({"done": list(done)}).encode("utf-8")

    blob.upload_blob(
        payload,
        overwrite=True,
        content_settings=ContentSettings(content_type="application/json"),
    )


# -----------------------
# HTTP (async) with retries/backoff
# -----------------------
async def fetch_image(session: aiohttp.ClientSession, pano_id: str, heading: int, sem: asyncio.Semaphore):
    url = "https://maps.googleapis.com/maps/api/streetview"
    params = {
        "size": image_size,
        "pano": pano_id,
        "heading": heading,
        "key": STREET_VIEW_KEY,
    }

    base_wait = 1.0
    for attempt in range(MAX_RETRIES):
        async with sem:
            try:
                async with session.get(url, params=params, timeout=ClientTimeout(total=REQUEST_TIMEOUT)) as r:
                    # success
                    if r.status == 200:
                        return await r.read()
                    # don't retry for client errors where resources won't appear
                    if r.status in (403, 404):
                        return None
                    # backoff on rate-limit or service issues
                    if r.status in (429, 503):
                        wait = min(60, base_wait * (2 ** attempt) + random.random())
                        await asyncio.sleep(wait)
                        continue
                    # other unexpected statuses: treat as non-retryable
                    return None
            except (aiohttp.ClientError, asyncio.TimeoutError):
                # transient network issue -> backoff and retry
                wait = min(60, base_wait * (2 ** attempt) + random.random())
                await asyncio.sleep(wait)
                continue
    return None

# -----------------------
# Per-metadata processing (unchanged semantics)
# -----------------------
async def process_metadata(meta_blob: str,
                           session: aiohttp.ClientSession,
                           sem: asyncio.Semaphore,
                           done: Set[str],
                           loop: asyncio.AbstractEventLoop,
                           pbar: tqdm,
                           processed_counter: dict):
    """Download metadata JSON, attempt 4 image downloads, upload each image."""
    if meta_blob in done:
        pbar.update(len(angles))
        processed_counter["count"] += 1
        return

    # Read metadata from Azure synchronously in executor (to avoid using azure aio client)
    def read_meta():
        bc = container_client.get_blob_client(meta_blob)
        raw = bc.download_blob().readall()
        return json.loads(raw)

    try:
        meta = await loop.run_in_executor(None, read_meta)
    except Exception:
        # Could not read meta — don't mark done so it can be retried later.
        # Update progress by 0 (we didn't process angles).
        return

    pano_id = meta.get("metadata_response", {}).get("pano_id")
    image_number = os.path.splitext(os.path.basename(meta_blob))[0]

    if not pano_id:
        # mark as done to avoid infinite retry (metadata broken)
        done.add(meta_blob)
        save_checkpoint(done)
        pbar.update(len(angles))
        processed_counter["count"] += 1
        return

    # schedule downloads and uploads for 4 angles
    async def dl_and_upload(heading):
        try:
            img = await fetch_image(session, pano_id, heading, sem)
            if not img:
                return False
            out_path = f"{prefix}/{target_folder}/{image_number}_{heading}.jpg"
            # run sync upload in executor
            uploaded = await loop.run_in_executor(None, upload_blob_sync, img, out_path)
            return bool(uploaded)
        except Exception:
            # upload or other error — treat as failure (we do not mark meta as not-done, so it can be retried later if you want)
            return False

    tasks = [asyncio.create_task(dl_and_upload(h)) for h in angles]
    # gather but do not let a single exception cancel everything
    results = await asyncio.gather(*tasks, return_exceptions=True)

    # You can choose to mark done even if some images failed; this avoids repeatedly reprocessing broken pano ids.
    done.add(meta_blob)
    save_checkpoint(done)

    # update progress bar by number of angles attempted (keeps progress consistent)
    pbar.update(len(angles))
    processed_counter["count"] += 1

# -----------------------
# Worker pattern: multiple metadata files in parallel
# -----------------------
async def meta_worker(name: int,
                      queue: asyncio.Queue,
                      session: aiohttp.ClientSession,
                      sem: asyncio.Semaphore,
                      done: Set[str],
                      loop: asyncio.AbstractEventLoop,
                      pbar: tqdm,
                      processed_counter: dict):
    try:
        while True:
            meta_blob = await queue.get()
            try:
                await process_metadata(meta_blob, session, sem, done, loop, pbar, processed_counter)
            finally:
                queue.task_done()
    except asyncio.CancelledError:
        # worker cancelled on shutdown
        return

# -----------------------
# Main async entry
# -----------------------
async def main_async():
    blobs = container_client.list_blobs(
        name_starts_with=f"{prefix}/{remote_meta_folder}"
    )
    meta_files = [b.name for b in blobs if b.name.endswith(".json")]

    total_meta = len(meta_files)
    total_images = total_meta * len(angles)
    print(f"Found {total_meta} metadata JSON files → up to {total_images} images")

    done = load_checkpoint()
    loop = asyncio.get_running_loop()

    connector = TCPConnector(limit=MAX_CONCURRENT_DOWNLOADS)
    timeout = ClientTimeout(total=None)
    sem = asyncio.Semaphore(MAX_CONCURRENT_DOWNLOADS)

    # Queue of metadata files
    q = asyncio.Queue()
    for m in meta_files:
        q.put_nowait(m)

    processed_counter = {"count": 0}

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
    # create progress bar BEFORE starting workers so they can reference it
      with tqdm(total=total_images, desc="Images", unit="img") as tqdm_bar:
          workers = [
              asyncio.create_task(meta_worker(i, q, session, sem, done, loop, tqdm_bar, processed_counter))
              for i in range(NUM_META_WORKERS)
          ]

          try:
              await q.join()

          except KeyboardInterrupt:
              print("Interrupted by user, cancelling workers…")
          finally:
              for w in workers:
                  w.cancel()
              await asyncio.gather(*workers, return_exceptions=True)
              save_checkpoint(done)

    print(
        f"Finished. Processed {processed_counter['count']} metadata files. "
        f"Checkpoint saved to {checkpoint_file}."
    )

await main_async()
